# Adaptive Confidence-Aware Readout Scheduling (Colab)

Notebook 10 extends Notebook 09 from fixed progressive schedules to **confidence-aware adaptive scheduling**.

Core question:

```text
Can model-confidence signals guide which sparse test queries to evaluate next,
so readout reaches target behavior with fewer evaluated queries?
```

Pipeline:

```text
many sparse test queries
    -> seed subset
    -> evaluate / score confidence
    -> prioritize uncertain queries
    -> progressive readout
    -> early stopping
```

Compared strategies:

1. deterministic mod30-style schedule
2. random schedule
3. confidence-aware adaptive schedule

Guardrail: this notebook evaluates classical readout scheduling behavior. It does **not** claim QOS improvement, quantum advantage, or model accuracy improvement.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train baseline classifier

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Confidence scoring

For `LinearSVC`, we use the decision function as a confidence proxy.

- larger margin = higher confidence
- smaller margin = lower confidence / more uncertainty

For multiclass classifiers, the margin proxy is the gap between the top two decision scores.


In [ ]:
def decision_margin_confidence(model, X):
    scores = model.decision_function(X)
    if scores.ndim == 1:
        return np.abs(scores)
    sorted_scores = np.sort(scores, axis=1)
    top = sorted_scores[:, -1]
    second = sorted_scores[:, -2]
    return top - second


confidence = decision_margin_confidence(clf, X_test)
confidence_summary = {
    "min": float(np.min(confidence)),
    "median": float(np.median(confidence)),
    "mean": float(np.mean(confidence)),
    "max": float(np.max(confidence)),
}
confidence_summary


## 3. Scheduling helpers

In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def evaluate_indices(indices):
    pred = clf.predict(X_test[indices])
    yt = y_test[indices]
    return {
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
        "mean_confidence": float(np.mean(confidence[indices])),
        "median_confidence": float(np.median(confidence[indices])),
    }


def progressive_curve(schedule_indices, fractions, schedule_name, schedule_type, trial=-1):
    rows = []
    total_queries = X_test.shape[0]
    for frac in fractions:
        k = max(2, int(np.ceil(len(schedule_indices) * frac)))
        idx = schedule_indices[:k]
        t0 = time.perf_counter()
        metrics = evaluate_indices(idx)
        eval_time = time.perf_counter() - t0
        rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            "schedule_fraction": frac,
            "n_eval": len(idx),
            "fraction_of_all_queries": len(idx) / total_queries,
            "eval_time_seconds": eval_time,
            **metrics,
        })
    return pd.DataFrame(rows)


fractions = np.linspace(0.05, 1.0, 20)
all_idx = np.arange(X_test.shape[0])
mod30_idx = wheel_query_indices(X_test.shape[0], "mod30")
len(mod30_idx), len(mod30_idx) / X_test.shape[0]


## 4. Define scheduling strategies

The adaptive strategy uses a deterministic seed, then prioritizes low-confidence examples.

This intentionally evaluates uncertain queries earlier. It controls *readout distribution*, not model training.


In [ ]:
def deterministic_mod_schedule(wheel_name="mod30"):
    return wheel_query_indices(X_test.shape[0], wheel_name)


def random_schedule(n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.array(rng.choice(all_idx, size=n_keep, replace=False), dtype=int)


def confidence_adaptive_schedule(seed_indices, total_keep=None, mode="low_confidence_first"):
    """Build an adaptive schedule: seed first, then remaining ordered by confidence."""
    if total_keep is None:
        total_keep = len(seed_indices)

    seed_indices = np.array(seed_indices, dtype=int)
    seed_set = set(seed_indices.tolist())
    remaining = np.array([i for i in all_idx if i not in seed_set], dtype=int)

    if mode == "low_confidence_first":
        ordered_remaining = remaining[np.argsort(confidence[remaining])]
    elif mode == "high_confidence_first":
        ordered_remaining = remaining[np.argsort(-confidence[remaining])]
    else:
        raise ValueError("mode must be low_confidence_first or high_confidence_first")

    full_schedule = np.concatenate([seed_indices, ordered_remaining])
    return full_schedule[:total_keep]


# Seed with first 10% of mod30 schedule, then adapt by low confidence.
seed_fraction = 0.10
seed_k = max(2, int(np.ceil(len(mod30_idx) * seed_fraction)))
seed_idx = mod30_idx[:seed_k]

adaptive_idx = confidence_adaptive_schedule(seed_idx, total_keep=len(mod30_idx), mode="low_confidence_first")
high_conf_idx = confidence_adaptive_schedule(seed_idx, total_keep=len(mod30_idx), mode="high_confidence_first")

len(seed_idx), len(adaptive_idx), adaptive_idx[:10]


## 5. Run progressive curves

In [ ]:
curve_rows = []

# Deterministic mod30 schedule.
curve_rows.append(progressive_curve(
    deterministic_mod_schedule("mod30"),
    fractions,
    schedule_name="mod30",
    schedule_type="deterministic",
    trial=-1,
))

# Confidence-aware adaptive schedule with same final query count as mod30.
curve_rows.append(progressive_curve(
    adaptive_idx,
    fractions,
    schedule_name="confidence_adaptive",
    schedule_type="adaptive_low_confidence",
    trial=-1,
))

# High-confidence-first contrast, useful as a diagnostic.
curve_rows.append(progressive_curve(
    high_conf_idx,
    fractions,
    schedule_name="high_confidence_first",
    schedule_type="adaptive_high_confidence",
    trial=-1,
))

# Random matched schedules.
for trial in range(N_RANDOM_TRIALS):
    ridx = random_schedule(len(mod30_idx), seed=RANDOM_STATE + 10000 + trial * 1009)
    curve_rows.append(progressive_curve(
        ridx,
        fractions,
        schedule_name="random_matched",
        schedule_type="random_matched",
        trial=trial,
    ))

curves_df = pd.concat(curve_rows, ignore_index=True)
curves_csv = data_dir / "10_confidence_adaptive_readout_curves.csv"
curves_df.to_csv(curves_csv, index=False)
curves_df.head()


## 6. Early stopping

In [ ]:
def stop_at_target(curve_df, target_fraction=0.95):
    target = target_fraction * full_balanced_accuracy
    for _, row in curve_df.iterrows():
        if row["balanced_accuracy"] >= target:
            return {
                "stop_reason": f"target_{target_fraction:.2f}",
                "target_balanced_accuracy": target,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"target_{target_fraction:.2f}_not_reached",
        "target_balanced_accuracy": target,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
    }


def stop_at_stability(curve_df, window=3, tolerance=0.01):
    vals = curve_df["balanced_accuracy"].to_numpy()
    for i in range(window, len(vals)):
        recent = vals[i-window:i+1]
        if np.max(recent) - np.min(recent) <= tolerance:
            row = curve_df.iloc[i]
            return {
                "stop_reason": f"stable_w{window}_tol{tolerance}",
                "target_balanced_accuracy": np.nan,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"stable_w{window}_tol{tolerance}_not_reached",
        "target_balanced_accuracy": np.nan,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
    }


stop_rows = []
for (schedule_name, schedule_type, trial), group in curves_df.groupby(["schedule_name", "schedule_type", "trial"]):
    group = group.sort_values("schedule_fraction")
    for rule_name, stop_result in [
        ("target_95", stop_at_target(group, target_fraction=0.95)),
        ("target_99", stop_at_target(group, target_fraction=0.99)),
        ("stability", stop_at_stability(group, window=3, tolerance=0.01)),
    ]:
        stop_rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            "rule": rule_name,
            **stop_result,
        })

stops_df = pd.DataFrame(stop_rows)
stops_csv = data_dir / "10_confidence_adaptive_readout_stops.csv"
stops_df.to_csv(stops_csv, index=False)
stops_df.head()


## 7. Summarize adaptive vs baselines

In [ ]:
random_stop_summary = (
    stops_df[stops_df["schedule_type"] == "random_matched"]
    .groupby("rule")
    .agg(
        random_stop_fraction_mean=("stop_fraction_of_all_queries", "mean"),
        random_stop_fraction_std=("stop_fraction_of_all_queries", "std"),
        random_stop_bal_acc_mean=("stop_balanced_accuracy", "mean"),
        random_stop_bal_acc_std=("stop_balanced_accuracy", "std"),
        random_stop_n_eval_mean=("stop_n_eval", "mean"),
        random_stop_n_eval_std=("stop_n_eval", "std"),
    )
    .reset_index()
)

det_stop = stops_df[stops_df["schedule_type"].isin([
    "deterministic",
    "adaptive_low_confidence",
    "adaptive_high_confidence",
])][[
    "schedule_name", "schedule_type", "rule", "stop_fraction_of_all_queries", "stop_balanced_accuracy", "stop_n_eval", "stop_reason"
]].rename(columns={
    "stop_fraction_of_all_queries": "stop_fraction",
    "stop_balanced_accuracy": "stop_bal_acc",
})

summary_df = det_stop.merge(random_stop_summary, on="rule")
summary_df["delta_stop_fraction_vs_random"] = summary_df["stop_fraction"] - summary_df["random_stop_fraction_mean"]
summary_df["delta_stop_bal_acc_vs_random"] = summary_df["stop_bal_acc"] - summary_df["random_stop_bal_acc_mean"]
summary_df["delta_stop_n_eval_vs_random"] = summary_df["stop_n_eval"] - summary_df["random_stop_n_eval_mean"]

summary_csv = data_dir / "10_confidence_adaptive_readout_summary.csv"
summary_df.to_csv(summary_csv, index=False)
summary_df


## 8. Figure 10a — progressive accuracy curves

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.2))

plot_types = [
    ("deterministic", "mod30"),
    ("adaptive_low_confidence", "confidence adaptive"),
    ("adaptive_high_confidence", "high-confidence contrast"),
]

for schedule_type, label in plot_types:
    df = curves_df[curves_df["schedule_type"] == schedule_type].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["balanced_accuracy"], marker="o", linewidth=2, label=label)

random_mean = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["balanced_accuracy"]
    .mean()
    .reset_index()
)
random_std = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["balanced_accuracy"]
    .std()
    .reset_index()
)

ax.plot(random_mean["fraction_of_all_queries"], random_mean["balanced_accuracy"], linewidth=2, label="random mean")
ax.fill_between(
    random_mean["fraction_of_all_queries"],
    random_mean["balanced_accuracy"] - random_std["balanced_accuracy"],
    random_mean["balanced_accuracy"] + random_std["balanced_accuracy"],
    alpha=0.2,
)

ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% target")
ax.set_title("Confidence-Aware Adaptive Readout Scheduling")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_a_svg = fig_dir / "figure_10a_confidence_adaptive_curves.svg"
fig_a_png = fig_dir / "figure_10a_confidence_adaptive_curves.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 9. Figure 10b — queries needed to reach 95% target

In [ ]:
target_summary = summary_df[summary_df["rule"] == "target_95"].copy()
target_summary = target_summary.set_index("schedule_type").loc[
    ["deterministic", "adaptive_low_confidence", "adaptive_high_confidence"]
].reset_index()

labels = ["mod30", "confidence adaptive", "high-confidence"]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(9.0, 5.2))
ax.bar(x, target_summary["stop_fraction"], label="deterministic/adaptive")
ax.axhline(
    target_summary["random_stop_fraction_mean"].iloc[0],
    linestyle="--",
    linewidth=1,
    label="random mean",
)
ax.set_title("Early Stop Fraction to Reach 95% of Full Balanced Accuracy")
ax.set_xlabel("Schedule")
ax.set_ylabel("Fraction of all test queries evaluated")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()

for i, value in enumerate(target_summary["stop_fraction"]):
    ax.text(i, value, f"{value:.3f}", ha="center", va="bottom")

fig.tight_layout()
fig_b_svg = fig_dir / "figure_10b_stop_fraction_95_target.svg"
fig_b_png = fig_dir / "figure_10b_stop_fraction_95_target.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 10. Figure 10c — delta vs random stop fraction

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.2))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(labels, target_summary["delta_stop_fraction_vs_random"])
ax.set_title("Stop Fraction Minus Random Mean (95% Target)")
ax.set_xlabel("Schedule")
ax.set_ylabel("Δ stop fraction")
ax.grid(True, axis="y", alpha=0.35)

for i, value in enumerate(target_summary["delta_stop_fraction_vs_random"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")

fig.tight_layout()
fig_c_svg = fig_dir / "figure_10c_stop_fraction_delta_vs_random.svg"
fig_c_png = fig_dir / "figure_10c_stop_fraction_delta_vs_random.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 11. Figure 10d — confidence profile of evaluated queries

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.2))

for schedule_type, label in plot_types:
    df = curves_df[curves_df["schedule_type"] == schedule_type].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["mean_confidence"], marker="o", linewidth=2, label=label)

ax.set_title("Mean Confidence of Evaluated Query Set")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Mean margin confidence")
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_d_svg = fig_dir / "figure_10d_confidence_profile.svg"
fig_d_png = fig_dir / "figure_10d_confidence_profile.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 12. Interpretation helper

In [ ]:
print("Full balanced accuracy:", full_balanced_accuracy)
print("95% target:", 0.95 * full_balanced_accuracy)
print()
display(summary_df)

for _, row in target_summary.iterrows():
    print(
        f"{row['schedule_name']} ({row['schedule_type']}): "
        f"stop_fraction={row['stop_fraction']:.3f}, "
        f"random_mean={row['random_stop_fraction_mean']:.3f}, "
        f"delta={row['delta_stop_fraction_vs_random']:+.3f}"
    )

print("""
Notebook 10 interpretation:

- Notebook 08 introduced deterministic readout schedules.
- Notebook 09 added early stopping.
- Notebook 10 adds confidence awareness: prioritize low-confidence queries after a deterministic seed.

Interpretation rules:
- If confidence-adaptive stop fraction < random mean, it reaches the target with fewer queries in this experiment.
- If approximately equal, value is structured and reproducible adaptive scheduling.
- If higher, confidence targeting may be less efficient for this dataset/model but still provides an interpretable control strategy.

Guardrail:
This is a classical readout scheduling experiment. It does not modify the model, QOS internals, or quantum hardware.
""")


## 13. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/10_confidence_adaptive_readout_curves.csv",
    "data/10_confidence_adaptive_readout_stops.csv",
    "data/10_confidence_adaptive_readout_summary.csv",
    "figures/figure_10a_confidence_adaptive_curves.svg",
    "figures/figure_10b_stop_fraction_95_target.svg",
    "figures/figure_10c_stop_fraction_delta_vs_random.svg",
    "figures/figure_10d_confidence_profile.svg",
]:
    files.download(path)
